# Copilot Audit Log Parser

Parses raw `CopilotInteraction` audit-log CSVs (with the nested `AuditData` JSON column) into a flat Delta table consumed by the **AI-in-One Dashboard** and **AI Business Value Dashboard** — Path D / Fabric variant.

**Inputs**: one or more raw audit CSVs landed in `Files/audit_raw/` of this Lakehouse. Expected columns: `RecordId, CreationDate, RecordType, Operation, AuditData, AssociatedAdminUnits, AssociatedAdminUnitsNames`.

**Output**: Lakehouse Delta table `dbo.Copilot_Interactions_Parsed` with the schema the PBIT expects (one row per prompt × accessed-resource).

**Schedule**: run on the same cadence as your audit-log export (typically daily). Configure via the **Schedule** button at the top of this notebook.


## 1. Configuration
Adjust paths if your raw CSVs land somewhere other than `Files/audit_raw/`.


In [4]:
# === CONFIG ===
RAW_PATH    = 'Files/audit_raw/*.csv'        # raw audit CSVs from the export script / pipeline
OUTPUT_TABLE = 'dbo.Copilot_Interactions_Parsed' # Delta table name (consumed by the PBIT)
WRITE_MODE  = 'overwrite'                    # use 'append' + a watermark column for incremental


StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 6, Finished, Available, Finished, False)

## 2. Define the AuditData JSON schema
Only the fields the PBIT consumes — adding more is harmless but slows the parse.


In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType
)

audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType())
        ]))),
    ])),
])

StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 7, Finished, Available, Finished, False)

## 3. Read raw CSVs and parse JSON


In [6]:
raw = (spark.read
    .option('header', 'true')
    .option('multiline', 'true')
    .option('escape', '"')
    .csv(RAW_PATH))

print(f'Raw rows: {raw.count():,}')

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))


StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 8, Finished, Available, Finished, False)

Raw rows: 376,556


In [14]:
# Inspect distinct isPrompt values and a sample Messages structure
from pyspark.sql import functions as F

print("Distinct isPrompt values:")
parsed.selectExpr("explode(j.CopilotEventData.Messages) as msg") \
      .select("msg.isPrompt") \
      .distinct() \
      .show(50, truncate=False)

print("Sample Messages structures:")
parsed.select(F.col("j.CopilotEventData.Messages").alias("Messages")).show(5, truncate=False)

StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 16, Finished, Available, Finished, False)

Distinct isPrompt values:
+--------+
|isPrompt|
+--------+
|false   |
|true    |
+--------+

Sample Messages structures:
+-----------------------------------------------------------------------+
|Messages                                                               |
+-----------------------------------------------------------------------+
|[{1749140036296, true}, {1749140036456, false}, {1749140036561, false}]|
|[{1748449582440, true}, {1748449582582, false}]                        |
|[{1742566321800, true}, {1742566321854, false}]                        |
|[{1748945344625, true}, {1748945344776, false}]                        |
|[{1744366239847, true}, {1744366239992, false}]                        |
+-----------------------------------------------------------------------+
only showing top 5 rows



## 4. Flatten + fan out per (prompt × resource)
Mirrors the existing AI-in-One M-query exactly — drop responses, expand resources.


In [17]:
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

# Flatten and fan out per (prompt × resource)
flat = (
    parsed
    .select(
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        F.col('j.CopilotEventData.ThreadId').alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        # Corrected bracket/parenthesis and keep first AISystemPlugin element
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
    # One row per message
    .withColumn('msg', F.explode('Messages'))
    # Your data has isPrompt as true/false; cast to string and compare case-insensitively
    .filter(F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
    # One row per accessed resource; if none, synthesize a single null resource
    .withColumn(
        'res',
        F.explode_outer(
            F.when(F.size('Resources') > 0, F.col('Resources'))
             .otherwise(F.array(F.struct(
                 F.lit(None).cast(StringType()).alias('Type'),
                 F.lit(None).cast(StringType()).alias('Action'),
                 F.lit(None).cast(StringType()).alias('SiteUrl'),
                 F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
             )))
        ),
    )
    .select(
        'CreationDate', 'AgentId', 'AgentName',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.col('msg.Id').alias('Message_Id'),
        F.col('msg.isPrompt').alias('Message_isPrompt'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
)

StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 19, Finished, Available, Finished, False)

## 5. Derive `Agent_TitleID`
Same logic as the M-query: pull the slug after the standard CopilotStudio prefix or before the first `.` for `P_` / `T_` declarative IDs.


In [18]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))


StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 20, Finished, Available, Finished, False)

## 6. Write to Lakehouse Delta table


In [19]:
(flat.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

print(f'Rows written to {OUTPUT_TABLE}: {spark.table(OUTPUT_TABLE).count():,}')

StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 21, Finished, Available, Finished, False)

Rows written to Copilot_Interactions_Parsed: 782,397


## 7. Verify
Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`, etc.


In [20]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)

StatementMeta(, 4f5617cc-9be1-4d31-be0d-4859fa4772f9, 22, Finished, Available, Finished, False)

+-----------------+--------+--------------+------+
|AISystemPlugin_Id|Workload|AppHost       |count |
+-----------------+--------+--------------+------+
|BingWebSearch    |Copilot |Teams         |327659|
|BingWebSearch    |Copilot |Office        |240731|
|NULL             |Copilot |Outlook       |71106 |
|NULL             |Copilot |Word          |34380 |
|BingWebSearch    |Copilot |Outlook       |23122 |
|NULL             |Copilot |Teams         |15682 |
|NULL             |Copilot |Unknown       |14632 |
|NULL             |Copilot |Office        |12773 |
|NULL             |Copilot |PowerPoint    |6630  |
|EnterpriseSearch |Copilot |Teams         |5533  |
|NULL             |Copilot |Excel         |5456  |
|NULL             |Copilot |Copilot Studio|4721  |
|NULL             |Copilot |Forms         |3279  |
|NULL             |Copilot |Designer      |3234  |
|BingWebSearch    |Copilot |Edge          |1649  |
|NULL             |Copilot |Autonomous    |1469  |
|NULL             |Copilot |One

---
**Connect the PBIT**: open the **Fabric variant** of the AI-in-One or AI Business Value Dashboard PBIT, supply two parameters when prompted:
- `Fabric SQL Endpoint` = `<workspace-guid>.datawarehouse.fabric.microsoft.com` (find in Lakehouse settings → SQL endpoint)
- `Lakehouse Database` = the lakehouse name (e.g. `CopilotAnalytics`)

The PBIT will source `dbo.Copilot_Interactions_Parsed` directly. Refresh becomes a near-instant data copy (or zero-copy with Direct Lake mode).
